In [1]:
import os

# Ensure TensorFlow uses the pure‑Python protobuf implementation (avoids import crash)
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPooling2D
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.preprocessing.image import (
    ImageDataGenerator,
    load_img,
    img_to_array,
)

print("Input directory listing:", os.listdir("../input"))




2026-05-08 13:07:23.953609: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778245643.969467      55 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778245643.974105      55 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-08 13:07:23.995639: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

Input directory listing: ['sample_submission.csv', 'description.md', 'test.zip', 'train.csv', 'sample_submission.csv.zip', 'test', 'train.zip', 'aerial-cactus-identification', 'train', 'train.csv.zip']


In [2]:
def read_pix(jpg_dir):
    """Read all .jpg files in a folder and return a NumPy array of shape (N,32,32,3)
    and a list of the corresponding filenames (without path)."""
    filenames = sorted(glob.glob(os.path.join(jpg_dir, "*.jpg")))
    n = len(filenames)
    img_array = np.zeros((n, 32, 32, 3), dtype=np.uint8)
    img_index = []
    for i, fname in enumerate(filenames):
        img = load_img(fname, target_size=(32, 32))
        arr = img_to_array(img)  # shape (32,32,3), dtype float32
        img_array[i] = arr.astype(np.uint8)
        img_index.append(os.path.basename(fname))
    return img_array, img_index


def prepare_data(img_array, img_index, train_response):
    """Split the data into training/validation sets keeping the image‑label alignment."""
    y_train_series, y_test_series = train_test_split(
        train_response,
        test_size=0.25,
        shuffle=True,
        random_state=12,
        stratify=train_response,
    )
    y_train = y_train_series.values[:, np.newaxis]
    y_test = y_test_series.values[:, np.newaxis]

    train_idx = [img_index.index(idx) for idx in y_train_series.index]
    test_idx = [img_index.index(idx) for idx in y_test_series.index]

    x_train = img_array[train_idx]
    x_test = img_array[test_idx]

    return x_train, y_train, x_test, y_test, y_train_series, y_test_series


def simple_oversample(x, y, oversample_class, oversample_factor=3):
    """Duplicate all samples of the given class a few times to reduce imbalance."""
    mask = np.ravel(y == oversample_class)
    x_ov = x[mask]
    y_ov = y[mask]
    for _ in range(oversample_factor):
        x = np.concatenate([x, x_ov], axis=0)
        y = np.concatenate([y, y_ov], axis=0)
    return x, y




In [3]:
BASE_PATH = "../input/aerial-cactus-identification"
TRAIN_IMG_PATH = os.path.join(BASE_PATH, "train")
TEST_IMG_PATH = os.path.join(BASE_PATH, "test")

train_labels = pd.read_csv(os.path.join(BASE_PATH, "train.csv"))
sample_submission = pd.read_csv(os.path.join(BASE_PATH, "sample_submission.csv"))

train_img_array, train_img_index = read_pix(TRAIN_IMG_PATH)
test_img_array, test_img_index = read_pix(TEST_IMG_PATH)

train_response = train_labels.set_index("id").loc[train_img_index, "has_cactus"]
train_response.index = train_img_index  # ensure index matches image filenames




In [4]:
x_train, y_train, x_test, y_test, y_train_series, y_test_series = prepare_data(
    train_img_array, train_img_index, train_response
)

# Oversample only the minority class (has_cactus = 1) in the training set
x_train, y_train = simple_oversample(
    x_train, y_train, oversample_class=1, oversample_factor=2
)

# Do NOT oversample the validation set to keep it unbiased
# (x_test, y_test remain as produced by prepare_data)

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0




In [5]:
train_datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.1,
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode="nearest",
)

train_generator = train_datagen.flow(x_train, y_train, batch_size=128)




In [6]:
def elaborate_model():
    model = Sequential()
    model.add(Conv2D(32, (5, 5), activation="relu", input_shape=(32, 32, 3)))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Conv2D(64, (5, 5), activation="relu"))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Conv2D(64, (3, 3), activation="relu"))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Flatten())
    model.add(Dropout(0.2))
    model.add(Dense(128, activation="relu"))
    model.add(Dense(1, activation="sigmoid"))
    model.compile(
        loss="binary_crossentropy",
        optimizer=RMSprop(learning_rate=1e-4),
        metrics=["accuracy"],
    )
    return model




In [7]:
model = elaborate_model()
history = model.fit(
    train_generator,
    steps_per_epoch=int(np.ceil(x_train.shape[0] / 128)),
    epochs=30,  # a few more epochs for better learning
    validation_data=(x_test, y_test),
    verbose=2,
)

val_acc = history.history["val_accuracy"][-1]
print(f"Validation accuracy after training: {val_acc:.4f}")




/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-05-08 13:07:41.329920: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1778245661.330382      55 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:06:00.0, compute capability: 8.6


Epoch 1/30


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
I0000 00:00:1778245662.899187     108 service.cc:148] XLA service 0x7f125c02bb20 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778245662.899229     108 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
2026-05-08 13:07:42.926337: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1778245663.027705     108 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1778245664.217051     108 device_compiler.h:188] Compile

208/208 - 12s - 56ms/step - accuracy: 0.8974 - loss: 0.3332 - val_accuracy: 0.7497 - val_loss: 0.6005
Epoch 2/30
208/208 - 8s - 37ms/step - accuracy: 0.9088 - loss: 0.2268 - val_accuracy: 0.8141 - val_loss: 0.4825
Epoch 3/30
208/208 - 8s - 37ms/step - accuracy: 0.9477 - loss: 0.1657 - val_accuracy: 0.9032 - val_loss: 0.2762
Epoch 4/30
208/208 - 8s - 37ms/step - accuracy: 0.9489 - loss: 0.1572 - val_accuracy: 0.9241 - val_loss: 0.2231
Epoch 5/30
208/208 - 8s - 37ms/step - accuracy: 0.9492 - loss: 0.1518 - val_accuracy: 0.8705 - val_loss: 0.3825
Epoch 6/30
208/208 - 8s - 37ms/step - accuracy: 0.9490 - loss: 0.1475 - val_accuracy: 0.8804 - val_loss: 0.3453
Epoch 7/30
208/208 - 8s - 36ms/step - accuracy: 0.9518 - loss: 0.1413 - val_accuracy: 0.8468 - val_loss: 0.4590
Epoch 8/30
208/208 - 8s - 38ms/step - accuracy: 0.9529 - loss: 0.1385 - val_accuracy: 0.8976 - val_loss: 0.2847
Epoch 9/30
208/208 - 8s - 37ms/step - accuracy: 0.9528 - loss: 0.1350 - val_accuracy: 0.8126 - val_loss: 0.6153
Ep

In [8]:
test_probs = model.predict(
    test_img_array.astype("float32") / 255.0, batch_size=256, verbose=0
).ravel()

submission = pd.DataFrame({"id": test_img_index, "has_cactus": test_probs})
submission.to_csv("submission.csv", index=False)
print("Submission file written to:", os.path.abspath("submission.csv"))

Submission file written to: /kaggle/working/submission.csv
